# Rugby Rulebook Cross-Reference ("Interdependency") Extraction

For each rulebook year, this notebook scans the cleaned text for
cross-references from one law to another (e.g. "see Law 12", "as in Law 4
(3)") and builds a directed, weighted graph — one node per law, an edge
`source -> target` weighted by how many times `source` references `target`
— then saves it as a `.gexf` file for later analysis.

Two variants are produced per year, going to two different output folders:
- **`laws/`** — references found only within the law text itself
- **`laws_plan_defs/`** — the same, plus references picked up from
  "definitions"/"plan" sections and phrases like "see the Plan"

As with the tree-extraction notebook, the exact regex used to detect a law
reference, and where "definitions" sections start/end, differs by
year-range because the rulebooks' formatting conventions changed over time.

## Requirements

```bash
pip install networkx numpy pyyaml
```
(`pyyaml`/`yaml` isn't actually used by any cell below despite being
imported — kept as originally written.)

## Data

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
├── interdependencies/
│   ├── laws/               (generated by this notebook)
│   └── laws_plan_defs/     (generated by this notebook)
└── text_files/
    └── cleaned_relavent_texts/
        ├── 1845/
        ├── ...
        └── 2018-2026/
```

Update `DATA_DIR` in the **Configuration** cell if your data lives
elsewhere. Only file paths were changed below — all regexes, variable
names, and graph-building logic are exactly as originally written.


## Imports

In [ ]:
import networkx as nx
from pathlib import Path 
import yaml
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout
import re
import string
import numpy as np
from collections import Counter

## Configuration

In [ ]:
DATA_DIR = Path('data')

TEXTS_DIR = DATA_DIR / 'text_files' / 'cleaned_relavent_texts'
INTERDEP_BASE = DATA_DIR / 'interdependencies'
LAWS_DIR = INTERDEP_BASE / 'laws'
LAWS_PLAN_DEFS_DIR = INTERDEP_BASE / 'laws_plan_defs'

## Reference-detection helpers

`law_identification_early`/`law_identification` scan a line for phrases like
"Law 12" or "Laws 4, 14" and extract the referenced law numbers (the
"_early" version is used for the oldest rulebooks, which phrase references
slightly differently — e.g. it also matches a reference at the very start
of a line, and has no "yard" exclusion). `plan_identification_early`/
`plan_identification` flag when a line mentions "the Plan" (encoded as a
reference to a synthetic node `0` for the early version, or `1` for the
later version). `law_identification_defs` is used only in the 1969-2002
"definitions" sections, which reference laws with a different punctuation
pattern (`Law 12 -`).

In [ ]:
def law_identification_early(source, source_dict, line):
    reference_strings = re.finditer('(Law|law|Laws|laws|Rule|Rules)\s([0-9]+)(.*?)(\.|(?!\((\w|\d))\)|\)\.)(\s|$)', line)
    for reference_string in reference_strings:
        if reference_string:
            numbers = reference_string.group(2) + reference_string.group(3)
            references = re.findall(r"(?<!\()(?<!\.)(?<!\.\w)(?<!\.\w\.)\b\d+", numbers)

            if source not in source_dict:
                source_dict[source] = []
            source_dict[source].extend(references)

    return source_dict

In [ ]:
def plan_identification_early(source, source_dict, line):
    if re.search('(the plan)|(the Plan)|(((See)|(see))\s((Plan)|(plan)))', line):
        if source not in source_dict:
            source_dict[source] = []
        source_dict[source].extend([0])

    return source_dict

In [ ]:
def law_identification(source, source_dict, line):
    reference_strings = re.finditer('(?!^)(Law|law|Laws|laws)\s([0-9]+)(.*?)(\.|(?!\((\w|\d))\)|\)\.)(\s|$)', line)
    for reference_string in reference_strings:
        if reference_string:
            numbers = reference_string.group(2) + reference_string.group(3)
            references = re.findall(r"(?<!\()(?<!\.)(?<!\.\w)(?<!\.\w\.)\b(\d+)(?!\s(?:yard))", numbers)

            if source not in source_dict:
                source_dict[source] = []
            source_dict[source].extend(references)

    return source_dict



In [ ]:
def plan_identification(source, source_dict, line):
    if re.search('(the plan)|(the Plan)|(((See)|(see))\s((Plan)|(plan)))', line):
        if source not in source_dict:
            source_dict[source] = []
        source_dict[source].extend([1])

    return source_dict

In [ ]:
def law_identification_defs(source, source_dict, line):
    reference_strings = re.finditer('(?!^)(Law|law|Laws|laws)\s([0-9]+)\s\-', line)
    for reference_string in reference_strings:
        if reference_string: 
            numbers = reference_string.group(2) 
            references = re.findall(r"\b\d+(?!\s*(?:metre|Metre))\b", numbers)

            if source not in source_dict:
                source_dict[source] = []
            source_dict[source].extend(references)

    return source_dict

Quick sanity check that `law_identification` extracts the expected law
numbers from a hand-written example line.

In [ ]:
law_identification(1, {1: []}, 'there are references to Laws 12, 14.b.1, 4 (3), 16.23, 13 (a), and 23. also law 2. ')

## 1874-1889

**Laws-only** version — parse each file and save the resulting graph.

In [ ]:
def interdependencies_1874_1889_original(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^(CASE LAW)', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification_early(real_law_no, source_dict, temp_text)

    
    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))

        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1874_1889 = (TEXTS_DIR / '1874-1889')
final_path_1874_1889 = LAWS_DIR

for file_path in initial_path_1874_1889.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1874_1889_original(file, year, final_path_1874_1889)

**Laws + Plan/Definitions** version of the same range(s).

In [ ]:
def interdependencies_1874_1889(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^(CASE LAW)', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification_early(real_law_no, source_dict, temp_text)
            source_dict = plan_identification_early(real_law_no, source_dict, temp_text)

    
    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))

        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)



In [ ]:
initial_path_1874_1889 = (TEXTS_DIR / '1874-1889')
final_path_1874_1889 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1874_1889.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1874_1889(file, year, final_path_1874_1889)

## 1894-1924

**Laws-only** version — parse each file and save the resulting graph.

In [ ]:
def interdependencies_1894_1924_original(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_directives = False


    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((INTERNATIONAL RUGBY FOOTBALL BOARD)|(INTERNATIONAL BOARD)|(At the Annual General Meeting))', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))

        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1894_1924 = (TEXTS_DIR / '1894-1924')
final_path_1894_1924 = LAWS_DIR

for file_path in initial_path_1894_1924.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1894_1924_original(file, year, final_path_1894_1924)

**Laws + Plan/Definitions** version of the same range(s).

In [ ]:
def interdependencies_1894_1924(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_directives = False


    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((INTERNATIONAL RUGBY FOOTBALL BOARD)|(INTERNATIONAL BOARD)|(At the Annual General Meeting))', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)
            source_dict = plan_identification_early(real_law_no, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))

        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1894_1924 = (TEXTS_DIR / '1894-1924')
final_path_1894_1924 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1894_1924.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1894_1924(file, year, final_path_1894_1924)

## 1929, 1934-1949, 1954-1959

One parser (per variant) is shared across all three ranges — only the input/output folders differ between runs.

**Laws-only** version — parse each file and save the resulting graph.

In [ ]:
def interdependencies_1929_1959_original(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_directives = False


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((CIRCULAR LETTER OF THE INTERNATIONAL RUGBY FOOTBALL BOARD)|(NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)
            source_dict = plan_identification(real_law_no, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1929 = (TEXTS_DIR / '1929')
final_path_1929 = LAWS_DIR

for file_path in initial_path_1929.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959_original(file, year, final_path_1929)

In [ ]:
initial_path_1934_1949 = (TEXTS_DIR / '1934-1949')
final_path_1934_1949 = LAWS_DIR

for file_path in initial_path_1934_1949.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959_original(file, year, final_path_1934_1949)

In [ ]:
initial_path_1954_1959 = (TEXTS_DIR / '1954-1959')
final_path_1954_1959 = LAWS_DIR

for file_path in initial_path_1954_1959.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959_original(file, year, final_path_1954_1959)

**Laws + Plan/Definitions** version of the same range(s).

In [ ]:
def interdependencies_1929_1959(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_directives = False


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((CIRCULAR LETTER OF THE INTERNATIONAL RUGBY FOOTBALL BOARD)|(NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
        
        if not in_directives and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)
            source_dict = plan_identification(real_law_no, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1929 = (TEXTS_DIR / '1929')
final_path_1929 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1929.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959(file, year, final_path_1929)

In [ ]:
initial_path_1934_1949 = (TEXTS_DIR / '1934-1949')
final_path_1934_1949 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1934_1949.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959(file, year, final_path_1934_1949)

In [ ]:
initial_path_1954_1959 = (TEXTS_DIR / '1954-1959')
final_path_1954_1959 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1954_1959.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1929_1959(file, year, final_path_1954_1959)

## 1969-1974, 1985-1995, 1997-1998, 2000-2002

One parser (per variant) is shared across all four ranges. The 'original' parser also treats 1969-1995 differently from 2000/2002 for where the definitions section starts/ends (see the `int(year)` checks inside).

**Laws-only** version — parse each file and save the resulting graph.

In [ ]:
def interdependencies_1969_2002_original(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_definitions = True


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            in_definitions = False
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((FOREWORD)|(INTERNATIONAL RUGBY FOOTBALL BOARD DIRECTIVES))', temp_text) and int(year)<=1995:
            in_directives = True
        
        if not in_directives and not in_definitions and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)

        # internal forewords
        if re.search('^((BEFORE THE MATCH)|(DURING THE MATCH)|(HOW THE GAME IS PLAYED)|(IN THE FIELD-OF-PLAY)|(RESTARTS)|(IN-GOAL))$', temp_text) and (int(year)==2000 or int(year)==2002):
            in_definitions = True

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1969_1974 = (TEXTS_DIR / '1969-1974')
final_path_1969_1974 = LAWS_DIR

for file_path in initial_path_1969_1974.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002_original(file, year, final_path_1969_1974)

In [ ]:
initial_path_1985_1995 = (TEXTS_DIR / '1985-1995')
final_path_1985_1995 = LAWS_DIR

for file_path in initial_path_1985_1995.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002_original(file, year, final_path_1985_1995)

In [ ]:
initial_path_1997_1998 = (TEXTS_DIR / '1997-1998')
final_path_1997_1998 = LAWS_DIR

for file_path in initial_path_1997_1998.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002_original(file, year, final_path_1997_1998)

In [ ]:
initial_path_2000_2002 = (TEXTS_DIR / '2000-2002')
final_path_2000_2002 = LAWS_DIR

for file_path in initial_path_2000_2002.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002_original(file, year, final_path_2000_2002)

**Laws + Plan/Definitions** version of the same range(s).

In [ ]:
def interdependencies_1969_2002(file, year, final_path):
    in_directives = False
    real_law_no = 0
    source_dict = {}
    in_definitions = True


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            in_definitions = False
            real_law_no += 1
            source_dict[real_law_no] = []

        # Ref notes
        if re.search('^((FOREWORD)|(INTERNATIONAL RUGBY FOOTBALL BOARD DIRECTIVES))', temp_text) and int(year)<=1995:
            in_directives = True
        
        if not in_directives and not in_definitions and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)

        if in_definitions:
            source_dict = law_identification_defs(0, source_dict, temp_text)

        # internal forewords
        if re.search('^((BEFORE THE MATCH)|(DURING THE MATCH)|(HOW THE GAME IS PLAYED)|(IN THE FIELD-OF-PLAY)|(RESTARTS)|(IN-GOAL))$', temp_text) and (int(year)==2000 or int(year)==2002):
            in_definitions = True

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_1969_1974 = (TEXTS_DIR / '1969-1974')
final_path_1969_1974 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1969_1974.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002(file, year, final_path_1969_1974)

In [ ]:
initial_path_1985_1995 = (TEXTS_DIR / '1985-1995')
final_path_1985_1995 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1985_1995.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002(file, year, final_path_1985_1995)

In [ ]:
initial_path_1997_1998 = (TEXTS_DIR / '1997-1998')
final_path_1997_1998 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_1997_1998.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002(file, year, final_path_1997_1998)

In [ ]:
initial_path_2000_2002 = (TEXTS_DIR / '2000-2002')
final_path_2000_2002 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_2000_2002.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_1969_2002(file, year, final_path_2000_2002)

## 2003-2016, 2018-2026

**Laws-only** version — parse each file and save the resulting graph.

In [ ]:
def interdependencies_2003_2026_original(file, year, final_path):
    real_law_no = 0
    source_dict = {}
    in_definitions = True


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW|Law)\s[0-9]', temp_text)):
            in_definitions = False
            real_law_no += 1
            source_dict[real_law_no] = []
        
        if not in_definitions and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_2003_2016 = (TEXTS_DIR / '2003-2016')
final_path_2003_2016 = LAWS_DIR

for file_path in initial_path_2003_2016.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_2003_2026_original(file, year, final_path_2003_2016)

In [ ]:
initial_path_2018_2026 = (TEXTS_DIR / '2018-2026')
final_path_2018_2026 = LAWS_DIR

for file_path in initial_path_2018_2026.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_2003_2026_original(file, year, final_path_2018_2026)

**Laws + Plan/Definitions** version of the same range(s).

In [ ]:
def interdependencies_2003_2026(file, year, final_path):
    real_law_no = 0
    source_dict = {}
    in_definitions = True


    for line in file:
        temp_text = line

         # returns laws 
        if (re.search(r'^(LAW|Law)\s[0-9]', temp_text)):
            in_definitions = False
            real_law_no += 1
            source_dict[real_law_no] = []
        
        if not in_definitions and real_law_no>=1:
            source_dict = law_identification(real_law_no, source_dict, temp_text)

        if in_definitions:
            source_dict = law_identification_defs(0, source_dict, temp_text)

    G = nx.DiGraph()

    for source, targets in source_dict.items():
        G.add_node(int(source))
        
        target_weights = Counter(targets)

        for target, weight in target_weights.items():
            G.add_edge(int(source), int(target), weight=weight)

    final_file = Path(str(final_path) + f'/{year}ints.gexf')
    print(final_file)

    nx.write_gexf(G, final_file)

In [ ]:
initial_path_2003_2016 = (TEXTS_DIR / '2003-2016')
final_path_2003_2016 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_2003_2016.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_2003_2026(file, year, final_path_2003_2016)

In [ ]:
initial_path_2018_2026 = (TEXTS_DIR / '2018-2026')
final_path_2018_2026 = LAWS_PLAN_DEFS_DIR

for file_path in initial_path_2018_2026.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        interdependencies_2003_2026(file, year, final_path_2018_2026)